[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab2_RAG_Evaluation_DeepEval_Student.ipynb)


# Lab 2: RAG Evaluation with DeepEval
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Your RAG system from Lab 1 looks like it works — but does it really? In clinical AI, confident wrong answers harm patients. Today you measure 4 critical RAG quality metrics on the same Wilms tumor RAG system.

### Objective
- Build test set of ground-truth clinical Q&A pairs
- Apply DeepEval's 4 core metrics
- Compare a 'bad' RAG vs 'good' RAG
- Identify common failure modes

---
## Setup

Install packages, load Colab secrets, then upload **WT.pdf**.


In [ ]:
!pip install -q llama-parse langchain langchain-openai langchain-community langchain-chroma chromadb deepeval

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')

In [ ]:
from google.colab import files
uploaded = files.upload()  # WT.pdf
PDF_PATH = '/content/WT.pdf'

---
## Cell 1 — Ground-truth test set

Define several **Wilms tumor** Q&A pairs. Each row: `input`, `expected_output`, and `retrieval_context` (short gold strings). These drive DeepEval and make regressions visible when you change chunking or `k`.


In [ ]:
# TODO: define 5 clinical Q&A pairs about Wilms tumor with expected answers.
# Each entry: {'input', 'expected_output', 'retrieval_context' (list[str] of ground-truth excerpts)}
test_cases = [
    {
        'input': 'What chemotherapy regimen is recommended for Stage IV Wilms tumor?',
        'expected_output': '...',
        'retrieval_context': ['...']
    },
    # TODO: add 4 more (Stage I FH, vincristine toxicity, anaplastic histology treatment, radiation indication)
]
print(f'{len(test_cases)} test cases defined')

---
## Parse PDF once (shared across both pipelines)

Both BAD and GOOD pipelines use the **same parsed markdown**, **same embedding model**, and **Chroma** — only **chunking** changes. That way metric differences mostly reflect retrieval quality, not parse noise.


In [ ]:
from llama_parse import LlamaParse
from langchain_core.documents import Document

parser = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown')
raw_docs = parser.load_data(PDF_PATH)
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw_docs]
print(f'Parsed {len(lc_docs)} docs')

---
## Cell 2 — BAD RAG (chunk_size=100, no overlap)

On purpose, chunks are **too small**: facts and dosing context get split across many segments, so retrieval often returns **incomplete** context. Expect weaker DeepEval scores.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from context.\n\n{context}'),
    ('human', '{input}')
])

# TODO: build bad pipeline with chunk_size=100, chunk_overlap=0
# Hint: bad_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_splitter = ...
bad_chunks = ...
bad_vs = Chroma.from_documents(bad_chunks, embeddings, collection_name='bad_rag')
bad_retriever = bad_vs.as_retriever(search_kwargs={'k': 4})
bad_chain = create_retrieval_chain(bad_retriever, create_stuff_documents_chain(llm, prompt))
print(f'Bad RAG: {len(bad_chunks)} tiny chunks')

---
## Cell 3 — GOOD RAG (chunk_size=1000, overlap 200)

Match Lab 1 defaults so each chunk usually holds a **coherent** paragraph or table fragment. Keep `source` metadata on each `Document` for traceability.


In [ ]:
# TODO: chunk_size=1000, chunk_overlap=200; preserve source metadata
good_splitter = ...
good_chunks = ...
good_vs = Chroma.from_documents(good_chunks, embeddings, collection_name='good_rag')
good_retriever = good_vs.as_retriever(search_kwargs={'k': 4})
good_chain = create_retrieval_chain(good_retriever, create_stuff_documents_chain(llm, prompt))
print(f'Good RAG: {len(good_chunks)} chunks')

---
## Cell 4 — DeepEval metrics

Instantiate **Faithfulness**, **Answer relevancy**, **Contextual relevancy**, **Contextual recall** with the same judge model (`gpt-4o-mini`). Read Lab 2 solutions markdown for plain-language definitions.


In [ ]:
from deepeval.metrics import (
    FaithfulnessMetric, AnswerRelevancyMetric,
    ContextualRelevancyMetric, ContextualRecallMetric,
)
from deepeval.test_case import LLMTestCase

# TODO: instantiate the 4 metrics with threshold=0.7, model='gpt-4o-mini'
faithfulness = ...
answer_relevancy = ...
ctx_relevancy = ...
ctx_recall = ...
metrics = [faithfulness, answer_relevancy, ctx_relevancy, ctx_recall]

## Cell 5 — Run evaluations on both pipelines

In [ ]:
def run_pipeline(chain, tc):
    res = chain.invoke({'input': tc['input']})
    return res['answer'], [d.page_content for d in res['context']]

def evaluate(chain, label):
    rows = []
    for tc in test_cases:
        # TODO: run pipeline → get actual_output and retrieval_context
        actual, ret_ctx = ...
        # TODO: build LLMTestCase
        case = LLMTestCase(
            input=tc['input'],
            actual_output=actual,
            expected_output=tc['expected_output'],
            retrieval_context=ret_ctx,
        )
        scores = {}
        for m in metrics:
            m.measure(case)
            scores[m.__class__.__name__] = m.score
        scores['question'] = tc['input'][:60]
        rows.append(scores)
    return rows

bad_results = evaluate(bad_chain, 'BAD')
good_results = evaluate(good_chain, 'GOOD')

## Cell 6 — Compare results

In [ ]:
import pandas as pd
df_bad = pd.DataFrame(bad_results)
df_good = pd.DataFrame(good_results)
print('BAD RAG mean scores:')
print(df_bad.drop(columns=['question']).mean())
print('\nGOOD RAG mean scores:')
print(df_good.drop(columns=['question']).mean())

## Cell 7 — Diagnose failures

In [ ]:
# TODO: for the lowest-scoring case in BAD RAG, print question + retrieved chunks
# Discuss: why did the metric fail? small chunks → loss of context
...

## Stretch — Add a custom G-Eval metric (e.g., 'clinical safety')

## KHCC Connection
Confident wrong answers harm patients. DeepEval makes RAG quality measurable so the AI Office can sign off on a clinical assistant before it sees real patients.